In [1]:
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.parallel import DistributedDataParallel, DataParallel
from utils import get_sigma_time, get_sample_time, VESDE, get_config
from model import UNet3DModel
import matplotlib.pyplot as plt
from torch_ema import ExponentialMovingAverage
import logging
import os
import sys
from os.path import join
import argparse

In [2]:
class VESDE_DDIM(VESDE):
    def __init__(self, sigma_min, sigma_max, N, T=1, eps=1e-5):
        super().__init__(sigma_min, sigma_max, N, T, eps)

    def get_ddim_schedule(self, num_inference_steps, method="quadratic"):
        """
        Generates time steps for the sampling process.
        Adapts the logic from the Keras simulation_ddim function.
        """
        if method == "linear":
            # Standard linear interpolation in time
            t_steps = torch.linspace(self.T, self.eps, num_inference_steps)
            
        elif method == "quadratic":
            # Quadratic interpolation in sqrt-time space
            # This matches: np.linspace(0, np.sqrt(T), n) ** 2
            # It concentrates steps near t=eps (where fine details form)
            
            # We interpolate linearly between sqrt(T) and sqrt(eps)
            sqrt_T = np.sqrt(self.T)
            sqrt_eps = np.sqrt(self.eps)
            
            # Create linear grid in sqrt space
            lin_grid = torch.linspace(sqrt_T, sqrt_eps, num_inference_steps)
            
            # Square it back to get the quadratic time schedule
            t_steps = lin_grid ** 2
            
        return t_steps.to(DEVICE)

    def ddim_step(self, x, t, t_prev, model_output, eta=0.0):
        """
        Deterministic DDIM update for Variance Exploding SDE.
        
        Args:
            x: Current state x_t
            t: Current time
            t_prev: Next time step (closer to 0)
            model_output: The raw output from the neural network
            eta: 0.0 for deterministic (ODE), >0 for stochasticity
        """
        # 1. Get noise scales (sigmas) for current and next step
        sigma_t = self.sigma_fn(t)[:, None, None, None, None]
        sigma_prev = self.sigma_fn(t_prev)[:, None, None, None, None]
        
        # 2. Extract predicted noise (epsilon)
        # In VE-SDE: Score = model_output / sigma_t
        # And Score ~= -epsilon / sigma_t
        # Therefore: model_output ~= -epsilon
        # So: epsilon_pred = -model_output
        eps_pred = -model_output

        # 3. Predict x_0 (clean data)
        # x_0 = x_t - sigma_t * epsilon
        x_0_pred = x - sigma_t * eps_pred
        
        # 4. Compute DDIM Variance (usually 0 for deterministic sampling)
        # This allows for interpolation between ODE (eta=0) and SDE (eta=1)
        sigma_tau = eta * torch.sqrt(
            (sigma_prev**2 / sigma_t**2) * (1 - (sigma_prev**2 / sigma_t**2)) # Simplified term for VE
        )
        # Note: For pure VE, strict DDIM sigma calculation is slightly different 
        # but for eta=0 (which is the goal of DDIM), the noise term vanishes anyway.
        
        # 5. Compute the direction to x_{t_prev}
        # Direction = sqrt(sigma_prev^2 - sigma_tau^2) * epsilon
        dir_xt = torch.sqrt(sigma_prev**2 - sigma_tau**2) * eps_pred
        
        # 6. Random noise (if eta > 0)
        noise = torch.randn_like(x) if eta > 0 else torch.zeros_like(x)
        
        # 7. Final Update
        x_prev = x_0_pred + dir_xt + sigma_tau * noise
        
        return x_prev, x_0_pred

In [3]:
from dataclasses import dataclass

@dataclass
class args:
    config = 'configs/standard_32.json'
    disable_tqdm = False

In [4]:
config_filename = args.config
enable_tqdm = not args.disable_tqdm
config = get_config(config_filename)

In [17]:
input_type = config.data.input_type
target_type = config.data.target_type

In [5]:
Nside = config.data.image_size
#DEVICE = config.device
DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')


sigma_time = get_sigma_time(config.model.sigma_min, config.model.sigma_max)
sample_time = get_sample_time(config.model.sampling_eps, config.model.T)

cosmo_dir = config.model.cosmo_dir
data_path = join(config.model.workdir, cosmo_dir)
checkpoint_dir = join(data_path, config.model.checkpoint_dir)

In [6]:
# Build pytorch dataloaders
input_data = np.float32(np.load(join(data_path, 'observation.npy')))
print("Loaded shape:", input_data.shape)
label_data = np.float32(np.load(join(data_path, 'truth.npy')))
input_data = torch.from_numpy(input_data).to(DEVICE)
label_data = torch.from_numpy(label_data).to(DEVICE)
input_data = torch.unsqueeze(input_data, dim=1)
label_data = torch.unsqueeze(label_data, dim=1)

Loaded shape: (1, 32, 32, 32)


In [7]:
# Initialize score model
model = UNet3DModel(config)
#model = DataParallel(model)
model = model.to(DEVICE)

ema = ExponentialMovingAverage(model.parameters(), decay=config.model.ema_rate)

sde = VESDE(config.model.sigma_min, config.model.sigma_max, config.model.num_scales, config.model.T, config.model.sampling_eps)

In [8]:
# Check for existing checkpoint
checkpoint_path = join(checkpoint_dir, 'checkpoint.pth')
if os.path.isfile(checkpoint_path):
    loaded_state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(loaded_state['model'], strict=False)
    ema.load_state_dict(loaded_state['ema'])
    logging.info(f"Loaded checkpoint from {checkpoint_path}.")
    print(f"Loaded checkpoint from {checkpoint_path}.")
else:
    logging.warning(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")
    print(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")

_ = model.eval()

Loaded checkpoint from run/standard_32/checkpoints/checkpoint.pth.


In [11]:
input_data = torch.tile(input_data, dims=(config.sampling.batch_size, 1, 1, 1, 1))
shape = (config.sampling.batch_size, 1, Nside, Nside, Nside)

In [25]:
# --- Configuration ---
# Set the number of steps you want (e.g., 50 is common for DDIM)
inference_steps = 100  
schedule_method = "quadratic" # Matches your Keras implementation

# Initialize the updated SDE class
sde_ddim = VESDE_DDIM(
    config.model.sigma_min, config.model.sigma_max, 
    config.model.num_scales, config.model.T, config.model.sampling_eps
)

# Generate the schedule
# Timesteps go from T (noisy) down to eps (clean)
timesteps = sde_ddim.get_ddim_schedule(inference_steps + 1, method=schedule_method)
timesteps = timesteps.to(DEVICE)

print(f'Sampling with DDIM ({schedule_method} schedule, {inference_steps} steps).')

samples = []

Sampling with DDIM (quadratic schedule, 100 steps).


In [15]:
from utils import get_filepath

data_root = '../Datasets'
model_folder = data_path

In [27]:
for sample_no in range(1900, 2000):
    print(f'Sampling {sample_no}')
    # z127_path = f"./Dataset/halo_LH_128/halo_lh_{sample_no}.npy" 
    z127_path = os.path.join(data_root, get_filepath(sample_no, target_type)) #  f"./Dataset/Train_z127_from_IC_2000/df_m_z=127_sim{sample_no}.npy"
    z0_path = os.path.join(data_root, get_filepath(sample_no, input_type)) # f"./Dataset/Train_z0_2000/{sample_no}_z0.npy"

    output_dir = os.path.join(
        model_folder, 'samples_', str(sample_no)
    )

    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    # === Load z=0 and add Gaussian noise ===
    N = config.data.image_size
    z0 = np.load(z0_path).reshape(N, N, N)
    noise_sigma = config.data.noise_sigma
    z0_noisy = z0 + noise_sigma * np.random.normal(size=z0.shape)
    z0_noisy = z0_noisy[np.newaxis, ...]  # shape: (1, 128, 128, 128)

    # === Load z=127 and normalize ===
    z127 = np.load(z127_path).reshape(N, N, N)
    z127_norm = (z127 - np.mean(z127)) / np.std(z127)
    z127_norm = z127_norm[np.newaxis, ...]

    np.save(os.path.join(output_dir, "truth.npy"), z127_norm)

    Nside = config.data.image_size
    #DEVICE = config.device

    sigma_time = get_sigma_time(config.model.sigma_min, config.model.sigma_max)
    sample_time = get_sample_time(config.model.sampling_eps, config.model.T)

    label_data = np.float32(np.load(join(output_dir, 'truth.npy')))
    input_data = torch.from_numpy(np.float32(z0_noisy)).to(DEVICE)
    label_data = torch.from_numpy(label_data).to(DEVICE)
    input_data = torch.unsqueeze(input_data, dim=1)
    label_data = torch.unsqueeze(label_data, dim=1)

    input_data = torch.tile(input_data, dims=(config.sampling.batch_size, 1, 1, 1, 1))
    shape = (config.sampling.batch_size, 1, Nside, Nside, Nside)

    timesteps = sde_ddim.get_ddim_schedule(inference_steps + 1, method=schedule_method)

    samples = []
    for j in tqdm(
        range(config.sampling.num_samples // config.sampling.batch_size),
        disable=False #  args.disable_tqdm
    ):
        with torch.no_grad(), ema.average_parameters():
            # Start with random noise
            x = sde_ddim.prior_sampling(shape).to(DEVICE)
            
            # Iterate through the schedule
            # We stop at len(timesteps) - 1 because we need a "next" step t_prev
            for i in tqdm(range(len(timesteps) - 1), disable=False): # args.disable_tqdm):
                t = timesteps[i]
                t_prev = timesteps[i+1]
                
                # Broadcast time to batch size
                t_vec = torch.ones(shape[0], device=DEVICE) * t
                t_prev_vec = torch.ones(shape[0], device=DEVICE) * t_prev
                
                # Run model
                # Note: Ensure your model accepts inputs in this order
                model_output = model(torch.cat([x, input_data], dim=1), t_vec)
                
                # DDIM Update (eta=0.0 for deterministic)
                x, x_mean = sde_ddim.ddim_step(x, t_vec, t_prev_vec, model_output, eta=0.0)

            # Store results
            samples.append(x.detach().cpu().numpy())
        
        # Save intermediate results
        np.save(join(output_dir, 'sample.npy'), np.array(samples))
        # print(f'Finished batch {j+1}')

    samples = np.array(samples).reshape(-1, Nside, Nside, Nside)
    np.save(os.path.join(output_dir, 'sample.npy'), samples)

Sampling 1900


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.70s/it]


Sampling 1901


100%|██████████| 1/1 [00:02<00:00,  2.69s/it]


...

100%|██████████| 1/1 [00:04<00:00,  4.66s/it]


Sampling 1998


100%|██████████| 1/1 [00:04<00:00,  4.45s/it]


Sampling 1999


100%|██████████| 1/1 [00:03<00:00,  3.94s/it]
